# 📝 Génération de résumés
Utilise le modèle entraîné pour résumer des articles en français.

## 1. Imports & Setup

In [ ]:
import sys
import os
import torch
import yaml

# Ajoute src au path
sys.path.append(os.path.join(os.getcwd(), 'src'))

from transformers import AutoTokenizer
from transformer import Transformer
from utils import summarize, load_last_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## 2. Chargement de la config

In [ ]:
with open('config/base.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

print('Config chargée :', cfg['model'])

## 3. Chargement du tokenizer & modèle

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg['tokenizer']['pretrained_model_name'])
cfg['model']['vocab_size'] = tokenizer.vocab_size

model = Transformer(**cfg['model']).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres : {n_params / 1e6:.1f} M')

## 4. Chargement du checkpoint

In [ ]:
checkpoint_dir = cfg['training']['checkpoint_dir']

# Charge le dernier checkpoint disponible
# (utilise des optimiseur/scheduler/scaler factices — on n'en a pas besoin pour l'inférence)
optimizer  = torch.optim.AdamW(model.parameters())
scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda _: 1)
scaler     = torch.amp.GradScaler(enabled=False)

start_epoch, last_loss = load_last_checkpoint(
    checkpoint_dir, model, optimizer, scheduler, scaler, device
)
print(f'Checkpoint epoch : {start_epoch - 1}  |  Loss : {last_loss:.4f}')

## 5. Génération d'un résumé

In [ ]:
article = """
Mettez ici votre article à résumer.
Le texte peut être long — il sera tronqué automatiquement selon max_src_len.
"""

max_len = cfg['data']['max_tgt_len']

resume = summarize(
    model=model,
    tokenizer=tokenizer,
    input_document=article.strip(),
    device=device,
    max_len=max_len
)

print('=== RÉSUMÉ GÉNÉRÉ ===')
print(resume)

## 6. Test sur des exemples MLSUM

In [ ]:
from datasets import load_dataset

data = load_dataset(
    cfg['dataset']['path'],
    cfg['dataset']['name'],
    trust_remote_code=True
)

# Prend les 5 premiers exemples du test set
test_samples = data[cfg['dataset'].get('test_split', 'test')].select(range(5))

for i, sample in enumerate(test_samples):
    article   = sample['text'].strip()
    reference = sample['summary'].strip()

    generated = summarize(
        model=model,
        tokenizer=tokenizer,
        input_document=article,
        device=device,
        max_len=max_len
    )

    print(f'\n{'='*60}')
    print(f'Exemple {i+1}')
    print(f'--- Article (extrait) ---')
    print(article[:300] + '...')
    print(f'--- Résumé de référence ---')
    print(reference)
    print(f'--- Résumé généré ---')
    print(generated)

## 7. Calcul des scores ROUGE sur N exemples

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

N = 50  # nombre d'exemples à évaluer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1, r2, rL = [], [], []
samples = data[cfg['dataset'].get('test_split', 'test')].select(range(N))

for sample in samples:
    generated = summarize(
        model=model,
        tokenizer=tokenizer,
        input_document=sample['text'].strip(),
        device=device,
        max_len=max_len
    )
    scores = scorer.score(sample['summary'].strip(), generated)
    r1.append(scores['rouge1'].fmeasure)
    r2.append(scores['rouge2'].fmeasure)
    rL.append(scores['rougeL'].fmeasure)

print(f'ROUGE-1 : {np.mean(r1):.4f}')
print(f'ROUGE-2 : {np.mean(r2):.4f}')
print(f'ROUGE-L : {np.mean(rL):.4f}')